In [1]:
import numpy as np
import pandas as pd

In [4]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

In [5]:
df = pd.read_csv('Datasets/D28/Covid_toy.csv')

In [8]:
df.sample(5)

,age,gender,fever,cough,city,has_covid
91,38,Male,NaN,Mild,Delhi,Yes
60,24,Female,102.0,Strong,Bangalore,Yes
37,55,Male,100.0,Mild,Kolkata,No
12,25,Female,99.0,Strong,Kolkata,No
29,34,Female,NaN,Strong,Mumbai,Yes


In [15]:
df['cough'].value_counts() # cough -> Ordinal Encoder

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [14]:
df['city'].value_counts() # city -> OneHotEncoder

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [13]:
df['gender'].value_counts() # gender -> OneHotEncoder

gender
Female    59
Male      41
Name: count, dtype: int64

In [20]:
df.isnull().sum() # fever -> SimpleImuter

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [25]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(df.drop(columns = ['has_covid']),df['has_covid'],test_size = 0.2)

In [28]:
X_train.shape

(80, 5)

In [29]:
X_test.shape

(20, 5)

# Without Column Tronsformer

## SimpleImputer
- fever

In [33]:
si = SimpleImputer()

X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.fit_transform(X_test[['fever']])

In [37]:
X_train_fever.shape

(80, 1)

In [38]:
X_test_fever.shape

(20, 1)

## OrdinalEncoder
- cough

In [41]:
oe = OrdinalEncoder(categories = [['Mild','Strong']])

X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.fit_transform(X_test[['cough']])

In [43]:
X_train_cough.shape

(80, 1)

In [44]:
X_test_cough.shape

(20, 1)

## OneHotEncoder
- gender
- city

In [65]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [71]:
X_train_gender_city[0:5,:]

array([[0., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 0.],
       [1., 0., 0., 1.]])

In [67]:
# Extracting Age

X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [68]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

In [70]:
X_train_transformed[0:5,:]

array([[ 19., 100.,   0.,   0.,   0.,   0.,   1.],
       [ 75., 104.,   0.,   1.,   0.,   0.,   1.],
       [ 40.,  98.,   0.,   1.,   0.,   0.,   1.],
       [ 22.,  99.,   0.,   0.,   0.,   0.,   0.],
       [ 42., 104.,   1.,   0.,   0.,   1.,   0.]])

# With Column Transformers

In [72]:
from sklearn.compose import ColumnTransformer

In [79]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(drop='first',sparse_output=False),['gender','city'])
],remainder='passthrough')

In [81]:
transformer.fit_transform(X_train).shape

(80, 7)

In [82]:
transformer.transform(X_test).shape

(20, 7)